# 01: 多块 PLS (MB-PLS) 分析
## 代谢组 + 转录组 + 蛋白组 三块对齐

本 notebook 演示 ChemoCalib 的核心建模能力：
1. 生成合成三组学数据
2. 训练 MB-PLS 模型
3. 提取 VIP 驱动代谢物
4. 分析块重要性与残差空间

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from chemocalib.models.mbpls import MultiBlockPLS, generate_toy_multiblock_data

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

In [ ]:
# 生成三块合成数据
blocks, y, feature_names = generate_toy_multiblock_data(
    n_samples=100,
    n_metabolites=50,
    n_transcripts=200,
    n_proteins=80,
    noise=0.1,
    seed=42,
)

print(f'代谢组 X1: {blocks[0].shape}')
print(f'转录组 X2: {blocks[1].shape}')
print(f'蛋白组 X3: {blocks[2].shape}')
print(f'响应 Y:   {y.shape}')

In [ ]:
# 训练 MB-PLS
model = MultiBlockPLS(
    n_components=5,
    block_names=['代谢组', '转录组', '蛋白组'],
)
model.fit(blocks, y)

print(model.summary())

In [ ]:
# 块重要性可视化
fig, ax = plt.subplots()
colors = ['#2ecc71', '#3498db', '#e74c3c']
bars = ax.bar(model.block_names, model.block_importance, color=colors)
ax.set_ylabel('Block Importance')
ax.set_title('MB-PLS Block Importance')
for bar, imp in zip(bars, model.block_importance):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{imp:.3f}', ha='center', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# VIP 驱动代谢物 (代谢组 Top 15)
drivers = model.get_driving_metabolites(block_idx=0, top_k=15)

fig, ax = plt.subplots()
indices = [f'met_{i}' for i in drivers['indices']]
ax.barh(range(len(indices)), drivers['vip_values'][::-1], color='#2ecc71')
ax.set_yticks(range(len(indices)))
ax.set_yticklabels(indices[::-1])
ax.set_xlabel('VIP Score')
ax.set_title('Top 15 Driving Metabolites (VIP)')
plt.tight_layout()
plt.show()

In [ ]:
# 残差不确定性分析
residuals = model.residual_space(blocks)
uncertainty = model.uncertainty_score(blocks)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (ax, R, name) in enumerate(zip(axes, residuals, model.block_names)):
    ax.hist(np.linalg.norm(R, axis=1), bins=30, color=colors[i], alpha=0.7, edgecolor='white')
    ax.set_title(f'{name} Residual Norm')
    ax.set_xlabel('||R||_2')
plt.tight_layout()
plt.show()

print(f'Top 10 不确定样本 (索引): {np.argsort(uncertainty)[::-1][:10]}')

In [ ]:
# 潜变量得分可视化 (前两个分量)
scores = model.super_scores
sc = plt.scatter(scores[:, 0], scores[:, 1], c=y, cmap='coolwarm', s=60, edgecolor='k')
plt.colorbar(sc, label='响应 Y')
plt.xlabel('Super Score LV1')
plt.ylabel('Super Score LV2')
plt.title('MB-PLS Super Scores (前两个潜变量)')
plt.tight_layout()
plt.show()